# Full Market-Simulator Initial Training

This Notebook performs the slow initial training of the complete market simulator.

It jointly trains:

I. the configurable Transformer market-interpretation encoder;

II. the K independent recurrent participant agents;

III. the learned softmax action gate;

IV. the stochastic six-target market-transition head.

This is the structural training run. The later daily agent-side fine-tuning workflow and the runtime on-board adaptation workflow belong in separate files.

## Recursive stochastic state design

The model predicts six transition variables: return, spread change, imbalance change, log total-depth change, log trade size, and log trade count. These reconstruct a complete reduced top-of-book state for the next second. Stochastic inference samples from the predicted Gaussian distribution, rebuilds the next feature row, and feeds it back recursively.

## Training scope

I. Jointly train the full simulator from the configured dataset.

II. Export the encoder, participant population, gate, and transition modules separately as well as in one full checkpoint.

III. Do not run daily participant fine-tuning or runtime on-board adaptation in this Notebook.

IV. Those later workflows should load the structural checkpoint and update only the approved agent-side modules.

## Aggressive 80 GB training profile

This version raises the physical batch size from 3,072 to 5,632 while reducing
the number of batches per epoch so that each epoch still sees approximately the
same number of sampled windows. It also increases DataLoader concurrency and
enables CUDA flash/memory-efficient attention backends.

Run `choose_largest_safe_batch()` once before training. It performs a genuine
forward/backward memory probe and selects the largest fitting value from
`(5632, 5120, 4608, 4096)`. Leave at least roughly 10–15 GB of reserved-memory
headroom because validation, checkpointing, CUDA workspaces, and allocator
fragmentation can temporarily increase usage.

## 1. Environment

In [ ]:
!pip -q install kaggle pyarrow pandas numpy torch tqdm scikit-learn matplotlib psutil

import os
os.environ.setdefault(
    "PYTORCH_CUDA_ALLOC_CONF",
    "expandable_segments:True,max_split_size_mb:256",
)

import math
import random
import hashlib
import time
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings(
    "ignore",
    message="enable_nested_tensor is True, but self.use_nested_tensor is False",
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

if torch.cuda.is_available():
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
AMP_DTYPE = (
    torch.bfloat16
    if DEVICE.type == "cuda"
    and torch.cuda.is_bf16_supported()
    else torch.float16
)

print("Device:", DEVICE)
print("Hardware:", GPU_NAME)
print("AMP dtype:", AMP_DTYPE)

## 2. Configuration

In [ ]:
@dataclass
class Config:
    initial_training_scope: str = "joint_full_model"
    later_daily_finetuning_scope: str = "agent_side_only"
    runtime_onboard_scope: str = "separate_runtime_file"

    # Data
    stock_limit: Optional[int] = 20
    dataset_fraction: float = 0.65
    fraction_seed: int = 42
    processed_schema_version: str = "optiver_1s_six_target_v2"

    # One-second sequence training
    warmup_len: int = 96
    prediction_len: int = 32

    # Encoder configuration
    encoder_type: str = "transformer"
    encoder_causal: bool = True
    decision_interval_seconds: float = 1.0
    external_feature_dim: int = 0

    d_model: int = 128
    latent_dim: int = 128
    nhead: int = 8
    layers: int = 4
    ff_dim: int = 256
    dropout: float = 0.10
    norm_first: bool = True

    # Independent recurrent agents
    # K is a hyperparameter; the current experiment uses K=24.
    num_agents: int = 24
    agent_hidden: int = 64
    action_dim: int = 4

    # Output dimensions
    target_dim: int = 6
    stochastic_temperature: float = 1.0
    minimum_spread: float = 1e-8
    minimum_total_depth: float = 0.0

    # Throughput
    batch_size: int = 3800
    gradient_accumulation_steps: int = 1
    num_workers: int = 12
    prefetch_factor: int = 3
    persistent_workers: bool = True
    pin_memory: bool = True

    use_amp: bool = True
    use_torch_compile: bool = False
    aggressive_batch_fallback: tuple = (5632, 5120, 4608, 4096)
    compile_mode: str = "reduce-overhead"

    # Epoch policy
    max_epochs: int = 30
    max_train_steps_per_epoch: Optional[int] = 889
    max_validation_batches: Optional[int] = 166
    max_test_batches: Optional[int] = None
    patience: int = 4
    min_delta: float = 1e-5

    # Optimization
    lr: float = 6e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0

    # Loss weights
    lambda_state: float = 1.0
    lambda_diversity: float = 0.03
    lambda_balance: float = 0.005

    # Scheduler
    scheduler_name: str = "onecycle"
    scheduler_factor: float = 0.5
    scheduler_patience: int = 2
    scheduler_min_lr: float = 1e-6
    cosine_t_max: int = 20
    cosine_eta_min: float = 1e-6
    step_size: int = 5
    step_gamma: float = 0.5
    onecycle_pct_start: float = 0.05
    onecycle_div_factor: float = 10.0
    onecycle_final_div_factor: float = 100.0

    return_zero_tolerance: float = 1e-12

    progress_update_every: int = 4

    @property
    def sequence_len(self) -> int:
        return self.warmup_len + self.prediction_len + 1

cfg = Config()
display(pd.Series(asdict(cfg), name="value").to_frame())
print("Total sequence length:", cfg.sequence_len)

## Design alignment notes

- `K = cfg.num_agents` is configurable; the current experiment uses 24.
- `cfg.encoder_causal` controls the Transformer attention mask. The current forecasting run uses causal attention.
- Agents receive the shared latent state and only their own previous hidden state/action. No same-step agent-to-agent communication is implemented.
- The implemented population mechanism is a learned softmax gate followed by weighted action aggregation.
- The transition head receives the weighted aggregate action only. No direct latent-state bypass.
- The optional external channel is structurally supported through `external_feature_dim`, but the uploaded Optiver dataset does not provide external features.
- Daily participant fine-tuning and runtime on-board adaptation are implemented in separate workflows.

## 3. Google Drive and dataset paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/bigalpha_market_project")
KAGGLE_JSON = DRIVE_PROJECT_ROOT / "credentials" / "kaggle.json"
DRIVE_DOWNLOAD_ROOT = DRIVE_PROJECT_ROOT / "data" / "downloads"
DRIVE_CHECKPOINT_ROOT = DRIVE_PROJECT_ROOT / "checkpoints"

DRIVE_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

ARCHIVE_PATH = (
    DRIVE_DOWNLOAD_ROOT
    / "optiver-realized-volatility-prediction.zip"
)

LOCAL_ROOT = Path("/content/optiver")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

BOOK_ROOT = LOCAL_ROOT / "book_train.parquet"
TRADE_ROOT = LOCAL_ROOT / "trade_train.parquet"

if not KAGGLE_JSON.exists():
    raise FileNotFoundError(
        f"Missing Kaggle credential: {KAGGLE_JSON}"
    )

!mkdir -p ~/.kaggle
!cp "{KAGGLE_JSON}" ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

if not ARCHIVE_PATH.exists():
    print("Downloading Optiver archive...")
    !kaggle competitions download \
        -c optiver-realized-volatility-prediction \
        -p "{DRIVE_DOWNLOAD_ROOT}"
else:
    print("Archive already exists in Drive.")

if not BOOK_ROOT.exists() or not TRADE_ROOT.exists():
    print("Extracting to local Colab storage...")
    !unzip -q -o "{ARCHIVE_PATH}" -d "{LOCAL_ROOT}"
else:
    print("Local extracted data already exists.")

print("BOOK_ROOT:", BOOK_ROOT)
print("TRADE_ROOT:", TRADE_ROOT)

## 4. One-second data preparation

Every bucket is resampled to seconds `0..599`.

Book columns are forward-filled. Trade values are aggregated per second and set
to zero when no trade occurs. We also preserve:

- `real_book_update`
- `trade_occurred`
- `seconds_since_last_real_update`
- `bucket_progress`

Targets are always the change from second `t` to second `t+1`.

## Optiver dataset contract

I. Raw book partitions must provide the ten standard Optiver book columns used by the model.

II. Raw trade partitions must provide `time_id`, `seconds_in_bucket`, `size`, and `order_count`.

III. Every raw bucket is resampled to seconds 0 through 599.

IV. Targets are computed before removing raw second 599, leaving 599 valid processed rows per bucket.

V. The model receives 19 normalized features and predicts six next-step targets.

VI. A 129-row sampled window contains 96 warmup positions, 32 supervised positions, and one final alignment row.


In [ ]:
EPS = 1e-12

FEATURES = [
    "bid_px_rel_1",
    "ask_px_rel_1",
    "log_bid_size_1",
    "log_ask_size_1",
    "bid_px_rel_2",
    "ask_px_rel_2",
    "log_bid_size_2",
    "log_ask_size_2",
    "spread",
    "imbalance_1",
    "return_1",
    "spread_change",
    "imbalance_change",
    "log_trade_size",
    "log_trade_count",
    "real_book_update",
    "trade_occurred",
    "seconds_since_last_real_update",
    "bucket_progress",
]

TARGETS = [
    "next_return",
    "next_spread_change",
    "next_imbalance_change",
    "next_log_total_depth_change",
    "next_log_trade_size",
    "next_log_trade_count",
]


def split_bucket(stock_id: int, time_id: int) -> str:
    value = int(
        hashlib.sha256(
            f"{stock_id}:{time_id}:one-second-v1".encode()
        ).hexdigest()[:8],
        16,
    ) / 0xFFFFFFFF

    if value < 0.70:
        return "train"
    if value < 0.85:
        return "validation"
    return "test"


def keep_bucket_fraction(
    stock_id: int,
    time_id: int,
    fraction: float,
) -> bool:
    if fraction >= 1.0:
        return True

    value = int(
        hashlib.sha256(
            (
                f"{stock_id}:{time_id}:"
                f"{cfg.fraction_seed}:fraction"
            ).encode()
        ).hexdigest()[:8],
        16,
    ) / 0xFFFFFFFF

    return value < fraction



OPTIVER_BOOK_COLUMNS = {
    "time_id",
    "seconds_in_bucket",
    "bid_price1",
    "ask_price1",
    "bid_price2",
    "ask_price2",
    "bid_size1",
    "ask_size1",
    "bid_size2",
    "ask_size2",
}

OPTIVER_TRADE_COLUMNS = {
    "time_id",
    "seconds_in_bucket",
    "size",
    "order_count",
}


def validate_optiver_raw_schema(
    book,
    trade,
    stock_id,
):
    missing_book = (
        OPTIVER_BOOK_COLUMNS
        - set(book.columns)
    )

    if missing_book:
        raise RuntimeError(
            f"Optiver book schema mismatch for stock_id={stock_id}. "
            "Missing columns: "
            + ", ".join(
                sorted(missing_book)
            )
        )

    if len(trade):
        missing_trade = (
            OPTIVER_TRADE_COLUMNS
            - set(trade.columns)
        )

        if missing_trade:
            raise RuntimeError(
                f"Optiver trade schema mismatch for stock_id={stock_id}. "
                "Missing columns: "
                + ", ".join(
                    sorted(missing_trade)
                )
            )


def load_raw_stock(stock_id: int):
    book = pd.read_parquet(
        BOOK_ROOT / f"stock_id={stock_id}"
    ).copy()

    book["stock_id"] = int(stock_id)

    trade_path = TRADE_ROOT / f"stock_id={stock_id}"

    if trade_path.exists():
        trade = pd.read_parquet(trade_path).copy()
    else:
        trade = pd.DataFrame(
            columns=[
                "time_id",
                "seconds_in_bucket",
                "size",
                "order_count",
            ]
        )

    validate_optiver_raw_schema(
        book,
        trade,
        stock_id,
    )

    return book, trade


def resample_bucket_to_one_second(
    book_bucket: pd.DataFrame,
    trade_bucket: pd.DataFrame,
    stock_id: int,
    time_id: int,
) -> pd.DataFrame:
    seconds = pd.DataFrame(
        {"seconds_in_bucket": np.arange(600, dtype=np.int16)}
    )

    book_bucket = (
        book_bucket
        .sort_values("seconds_in_bucket")
        .drop_duplicates(
            subset=["seconds_in_bucket"],
            keep="last",
        )
        .copy()
    )

    book_bucket["real_book_update"] = 1.0

    frame = seconds.merge(
        book_bucket,
        on="seconds_in_bucket",
        how="left",
    )

    frame["real_book_update"] = (
        frame["real_book_update"]
        .fillna(0.0)
        .astype(np.float32)
    )

    book_columns = [
        "bid_price1",
        "ask_price1",
        "bid_price2",
        "ask_price2",
        "bid_size1",
        "ask_size1",
        "bid_size2",
        "ask_size2",
    ]

    frame[book_columns] = (
        frame[book_columns]
        .ffill()
        .bfill()
    )

    if len(trade_bucket):
        trade_per_second = (
            trade_bucket
            .groupby(
                "seconds_in_bucket",
                as_index=False,
            )
            .agg(
                trade_size=("size", "sum"),
                trade_count=("order_count", "sum"),
            )
        )

        frame = frame.merge(
            trade_per_second,
            on="seconds_in_bucket",
            how="left",
        )
    else:
        frame["trade_size"] = 0.0
        frame["trade_count"] = 0.0

    frame["trade_size"] = (
        frame["trade_size"]
        .fillna(0.0)
    )

    frame["trade_count"] = (
        frame["trade_count"]
        .fillna(0.0)
    )

    frame["trade_occurred"] = (
        frame["trade_count"] > 0
    ).astype(np.float32)

    last_update_second = (
        frame["seconds_in_bucket"]
        .where(
            frame["real_book_update"] > 0
        )
        .ffill()
        .fillna(0)
    )

    frame["seconds_since_last_real_update"] = (
        frame["seconds_in_bucket"]
        - last_update_second
    ).astype(np.float32)

    frame["stock_id"] = int(stock_id)
    frame["time_id"] = int(time_id)
    frame["bucket_progress"] = (
        frame["seconds_in_bucket"] / 599.0
    ).astype(np.float32)

    mid = (
        frame["bid_price1"]
        + frame["ask_price1"]
    ) / 2.0

    frame["mid_price"] = mid

    for level in (1, 2):
        frame[f"bid_px_rel_{level}"] = (
            frame[f"bid_price{level}"] - mid
        ) / mid.clip(lower=EPS)

        frame[f"ask_px_rel_{level}"] = (
            frame[f"ask_price{level}"] - mid
        ) / mid.clip(lower=EPS)

        frame[f"log_bid_size_{level}"] = np.log1p(
            frame[f"bid_size{level}"]
        )

        frame[f"log_ask_size_{level}"] = np.log1p(
            frame[f"ask_size{level}"]
        )

    frame["spread"] = (
        frame["ask_price1"]
        - frame["bid_price1"]
    )

    frame["imbalance_1"] = (
        frame["bid_size1"]
        - frame["ask_size1"]
    ) / (
        frame["bid_size1"]
        + frame["ask_size1"]
    ).clip(lower=EPS)

    frame["log_trade_size"] = np.log1p(
        frame["trade_size"]
    )

    frame["log_trade_count"] = np.log1p(
        frame["trade_count"]
    )

    frame["log_mid"] = np.log(
        frame["mid_price"].clip(lower=EPS)
    )

    frame["return_1"] = (
        frame["log_mid"]
        .diff()
        .fillna(0.0)
    )

    frame["spread_change"] = (
        frame["spread"]
        .diff()
        .fillna(0.0)
    )

    frame["imbalance_change"] = (
        frame["imbalance_1"]
        .diff()
        .fillna(0.0)
    )

    frame["total_depth_1"] = (
        frame["bid_size1"]
        + frame["ask_size1"]
    ).clip(lower=0.0)

    frame["log_total_depth_1"] = np.log1p(
        frame["total_depth_1"]
    )

    frame["next_return"] = (
        frame["return_1"].shift(-1)
    )

    frame["next_spread_change"] = (
        frame["spread_change"].shift(-1)
    )

    frame["next_imbalance_change"] = (
        frame["imbalance_change"].shift(-1)
    )

    frame["next_log_total_depth_change"] = (
        frame["log_total_depth_1"].shift(-1)
        - frame["log_total_depth_1"]
    )

    frame["next_log_trade_size"] = (
        frame["log_trade_size"].shift(-1)
    )

    frame["next_log_trade_count"] = (
        frame["log_trade_count"].shift(-1)
    )


    return (
        frame.iloc[:-1]
        .reset_index(drop=True)
    )


def prepare_one_second_frame():
    stock_ids = sorted(
        int(path.name.split("=")[1])
        for path in BOOK_ROOT.glob("stock_id=*")
    )

    if cfg.stock_limit is not None:
        stock_ids = stock_ids[:cfg.stock_limit]

    frames = []

    for stock_id in tqdm(
        stock_ids,
        desc="Preparing one-second stocks",
        dynamic_ncols=True,
    ):
        book, trade = load_raw_stock(stock_id)

        time_ids = sorted(
            book["time_id"].unique().tolist()
        )

        for time_id in time_ids:
            if not keep_bucket_fraction(
                stock_id,
                int(time_id),
                cfg.dataset_fraction,
            ):
                continue

            book_bucket = book.loc[
                book["time_id"] == time_id
            ]

            trade_bucket = trade.loc[
                trade["time_id"] == time_id
            ] if len(trade) else trade

            bucket = resample_bucket_to_one_second(
                book_bucket,
                trade_bucket,
                stock_id,
                int(time_id),
            )

            bucket["split"] = split_bucket(
                stock_id,
                int(time_id),
            )

            frames.append(bucket)

    frame = pd.concat(
        frames,
        ignore_index=True,
    )

    frame = (
        frame.sort_values(
            [
                "stock_id",
                "time_id",
                "seconds_in_bucket",
            ]
        )
        .reset_index(drop=True)
    )

    return frame

## 5. Prepare all data before training

In [ ]:
prepare_start = time.time()

FRAME = prepare_one_second_frame()

print("Rows:", f"{len(FRAME):,}")
print("Stocks:", FRAME["stock_id"].nunique())
print(
    "Buckets:",
    FRAME[
        ["stock_id", "time_id"]
    ].drop_duplicates().shape[0],
)

display(
    FRAME["split"]
    .value_counts()
    .rename("rows")
    .to_frame()
)

print(
    "Real update fraction:",
    f"{FRAME['real_book_update'].mean():.2%}",
)

print(
    "Preparation minutes:",
    round(
        (time.time() - prepare_start) / 60,
        2,
    ),
)

In [ ]:
PROCESSED_DATASET_ROOT = (
    DRIVE_PROJECT_ROOT
    / "data"
    / "processed"
    / cfg.processed_schema_version
)

PROCESSED_DATASET_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

save_start = time.time()

FRAME.to_parquet(
    PROCESSED_DATASET_ROOT,
    engine="pyarrow",
    compression="zstd",
    index=False,
    partition_cols=[
        "split",
        "stock_id",
    ],
)

print(
    "Saved partitioned dataset:",
    PROCESSED_DATASET_ROOT,
)

print(
    "Saving minutes:",
    round(
        (time.time() - save_start) / 60,
        2,
    ),
)

Load preprocessed files

In [ ]:
PROCESSED_DATASET_ROOT = (
    DRIVE_PROJECT_ROOT
    / "data"
    / "processed"
    / cfg.processed_schema_version
)

FRAME = pd.read_parquet(
    PROCESSED_DATASET_ROOT,
    engine="pyarrow",
)
TRAIN_FRAME = pd.read_parquet(
    PROCESSED_DATASET_ROOT,
    engine="pyarrow",
    filters=[
        ("split", "==", "train"),
    ],
)

## 6. First stock, first three10 one-second rows

## Processed Optiver schema validation

The current preprocessing computes `next_log_total_depth_change` from all 600 raw seconds and only then removes the final raw row. Each processed bucket must therefore retain 599 valid supervised rows.

This cell validates the loaded partitioned dataset. It does not recompute the target from an already truncated cache.

In [ ]:
if "FRAME" not in globals():
    raise RuntimeError(
        "FRAME is not loaded. Run the processed-data loading cell first."
    )

REQUIRED_PROCESSED_COLUMNS = {
    "stock_id",
    "time_id",
    "seconds_in_bucket",
    "split",
    "bid_size1",
    "ask_size1",
    "total_depth_1",
    "log_total_depth_1",
    *FEATURES,
    *TARGETS,
}

missing_processed_columns = (
    REQUIRED_PROCESSED_COLUMNS
    - set(FRAME.columns)
)

if missing_processed_columns:
    raise RuntimeError(
        "The processed Optiver dataset does not match the current "
        f"schema version {cfg.processed_schema_version}. Missing: "
        + ", ".join(
            sorted(missing_processed_columns)
        )
        + ". Rebuild the processed dataset from the raw Optiver files."
    )

FRAME = (
    FRAME.sort_values(
        [
            "stock_id",
            "time_id",
            "seconds_in_bucket",
        ]
    )
    .reset_index(drop=True)
)

nonfinite_columns = {}

for column in (
    list(FEATURES)
    + list(TARGETS)
):
    values = pd.to_numeric(
        FRAME[column],
        errors="coerce",
    ).to_numpy(
        dtype=np.float64
    )

    count = int(
        (~np.isfinite(values)).sum()
    )

    if count:
        nonfinite_columns[column] = count

if nonfinite_columns:
    raise RuntimeError(
        "Non-finite values remain in the processed Optiver dataset: "
        + str(nonfinite_columns)
    )

bucket_sizes = (
    FRAME.groupby(
        [
            "stock_id",
            "time_id",
        ],
        observed=True,
    )
    .size()
)

invalid_bucket_sizes = (
    bucket_sizes[
        bucket_sizes != 599
    ]
)

if len(invalid_bucket_sizes):
    raise RuntimeError(
        "Processed Optiver buckets must contain exactly 599 rows. "
        f"Found {len(invalid_bucket_sizes):,} incompatible buckets. "
        "Rebuild using the current preprocessing cell rather than "
        "repairing a truncated cache."
    )

print(
    "Processed Optiver schema validated:",
    cfg.processed_schema_version,
)

print(
    "Rows per bucket:",
    int(bucket_sizes.iloc[0]),
)

print(
    "Buckets:",
    f"{len(bucket_sizes):,}",
)

In [ ]:
required_preview_columns = {
    "total_depth_1",
    "log_total_depth_1",
    "next_log_total_depth_change",
}

missing_preview_columns = (
    required_preview_columns
    - set(FRAME.columns)
)

if missing_preview_columns:
    raise RuntimeError(
        "FRAME was loaded from an outdated processed cache. "
        "Rerun the processed-data loading cell with "
        "FORCE_REBUILD_PROCESSED = True. Missing: "
        + ", ".join(
            sorted(missing_preview_columns)
        )
    )

first_stock_id = int(
    FRAME["stock_id"]
    .astype(int)
    .min()
)

first_stock = FRAME.loc[
    FRAME["stock_id"].astype(int)
    == first_stock_id
]

first_time_id = int(
    first_stock["time_id"]
    .astype(int)
    .min()
)

preview_columns = [
    "stock_id",
    "time_id",
    "seconds_in_bucket",
    "real_book_update",
    "seconds_since_last_real_update",
    "bid_price1",
    "ask_price1",
    "bid_size1",
    "ask_size1",
    "mid_price",
    "spread",
    "imbalance_1",
    "total_depth_1",
    "log_total_depth_1",
    "trade_size",
    "trade_count",
    "next_return",
    "next_spread_change",
    "next_imbalance_change",
    "next_log_total_depth_change",
]

FIRST_THREE10_ROWS = (
    first_stock.loc[
        first_stock["time_id"].astype(int)
        == first_time_id,
        preview_columns,
    ]
    .head(30)
    .reset_index(drop=True)
)

display(FIRST_THREE10_ROWS)

## 7. Sequence dataset

Each sample is one consecutive sequence from one stock and one bucket.

The first `warmup_len` seconds establish Transformer context and agent memory.
Loss is applied only over the following `prediction_len` seconds.

In [ ]:
class OneSecondSequenceDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        split: str,
        stats=None,
    ):
        self.frame = (
            frame.loc[
                frame["split"] == split
            ]
            .sort_values(
                [
                    "stock_id",
                    "time_id",
                    "seconds_in_bucket",
                ]
            )
            .reset_index(drop=True)
        )

        if stats is None:
            x_values = (
                self.frame[FEATURES]
                .replace(
                    [np.inf, -np.inf],
                    np.nan,
                )
                .astype(np.float32)
            )

            y_values = (
                self.frame[TARGETS]
                .replace(
                    [np.inf, -np.inf],
                    np.nan,
                )
                .astype(np.float32)
            )

            if (
                x_values.isna().any().any()
                or y_values.isna().any().any()
            ):
                bad_x = (
                    x_values.columns[
                        x_values.isna().any()
                    ].tolist()
                )

                bad_y = (
                    y_values.columns[
                        y_values.isna().any()
                    ].tolist()
                )

                raise RuntimeError(
                    "Non-finite values found before normalization. "
                    f"Bad FEATURES={bad_x}; "
                    f"bad TARGETS={bad_y}"
                )

            x_mean = (
                x_values.mean()
                .to_numpy(np.float32)
            )

            x_std = (
                x_values.std()
                .to_numpy(np.float32)
            )

            y_mean = (
                y_values.mean()
                .to_numpy(np.float32)
            )

            y_std = (
                y_values.std()
                .to_numpy(np.float32)
            )

            x_std = np.where(
                np.isfinite(x_std)
                & (x_std > 1e-6),
                x_std,
                1.0,
            ).astype(np.float32)

            y_std = np.where(
                np.isfinite(y_std)
                & (y_std > 1e-8),
                y_std,
                1.0,
            ).astype(np.float32)

            stats = {
                "x_mean": x_mean,
                "x_std": x_std,
                "y_mean": y_mean,
                "y_std": y_std,
            }

        self.stats = stats

        raw_x = self.frame[
            FEATURES
        ].to_numpy(np.float32)

        raw_y = self.frame[
            TARGETS
        ].to_numpy(np.float32)

        self.x_data = (
            raw_x - stats["x_mean"]
        ) / stats["x_std"]

        self.y_data = (
            raw_y - stats["y_mean"]
        ) / stats["y_std"]


        starts = []
        counts = []

        grouped = self.frame.groupby(
            ["stock_id", "time_id"],
            sort=False,
            observed=True,
        )

        for _, positions in tqdm(
            grouped.indices.items(),
            total=grouped.ngroups,
            desc=f"Indexing {split} sequences",
            dynamic_ncols=True,
        ):
            positions = np.asarray(
                positions,
                dtype=np.int64,
            )

            count = (
                len(positions)
                - cfg.sequence_len
                + 1
            )

            if count > 0:
                starts.append(int(positions[0]))
                counts.append(int(count))

        if not counts:
            raise ValueError(
                f"No valid sequences for split={split}"
            )

        self.starts = np.asarray(
            starts,
            dtype=np.int64,
        )

        if not np.isfinite(
            self.x_data
        ).all():
            raise RuntimeError(
                "Normalized dataset contains non-finite FEATURES."
            )

        if not np.isfinite(
            self.y_data
        ).all():
            raise RuntimeError(
                "Normalized dataset contains non-finite TARGETS."
            )

        self.counts = np.asarray(
            counts,
            dtype=np.int64,
        )

        self.cumulative = np.cumsum(
            self.counts
        )

        self.total_samples = int(
            self.cumulative[-1]
        )

        print(
            f"{split}: rows={len(self.frame):,}, "
            f"buckets={len(self.starts):,}, "
            f"samples={self.total_samples:,}"
        )

    def __len__(self):
        return self.total_samples

    def __getitem__(self, index):
        group_index = int(
            np.searchsorted(
                self.cumulative,
                index,
                side="right",
            )
        )

        previous = (
            0
            if group_index == 0
            else int(
                self.cumulative[
                    group_index - 1
                ]
            )
        )

        local_index = index - previous

        start = int(
            self.starts[group_index]
            + local_index
        )

        end = start + cfg.sequence_len

        return (
            torch.from_numpy(
                self.x_data[start:end]
            ),
            torch.from_numpy(
                self.y_data[start:end]
            ),
        )

## 8. Build datasets and DataLoaders

In [ ]:
TRAIN_DATASET = OneSecondSequenceDataset(
    FRAME,
    "train",
)

VALIDATION_DATASET = OneSecondSequenceDataset(
    FRAME,
    "validation",
    stats=TRAIN_DATASET.stats,
)

TEST_DATASET = OneSecondSequenceDataset(
    FRAME,
    "test",
    stats=TRAIN_DATASET.stats,
)


def loader_kwargs():
    values = {
        "num_workers": cfg.num_workers,
        "pin_memory": (
            cfg.pin_memory
            and DEVICE.type == "cuda"
        ),
        "persistent_workers": (
            cfg.persistent_workers
            and cfg.num_workers > 0
        ),
    }

    if cfg.num_workers > 0:
        values["prefetch_factor"] = (
            cfg.prefetch_factor
        )

    return values


def build_data_loaders(
    batch_size=None,
):
    """
    Build train, validation, and test loaders for the selected batch size.

    Batch size affects GPU memory only; it does not change the Optiver
    feature schema, targets, bucket boundaries, or split assignments.
    """
    if batch_size is None:
        batch_size = cfg.batch_size

    train_loader = DataLoader(
        TRAIN_DATASET,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        **loader_kwargs(),
    )

    validation_loader = DataLoader(
        VALIDATION_DATASET,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        **loader_kwargs(),
    )

    test_loader = DataLoader(
        TEST_DATASET,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        **loader_kwargs(),
    )

    train_steps = (
        len(train_loader)
        if cfg.max_train_steps_per_epoch
        is None
        else min(
            len(train_loader),
            cfg.max_train_steps_per_epoch,
        )
    )

    validation_steps = (
        len(validation_loader)
        if cfg.max_validation_batches
        is None
        else min(
            len(validation_loader),
            cfg.max_validation_batches,
        )
    )

    test_steps = (
        len(test_loader)
        if cfg.max_test_batches
        is None
        else min(
            len(test_loader),
            cfg.max_test_batches,
        )
    )

    return (
        train_loader,
        validation_loader,
        test_loader,
        train_steps,
        validation_steps,
        test_steps,
    )


(
    TRAIN_LOADER,
    VALIDATION_LOADER,
    TEST_LOADER,
    TRAIN_STEPS,
    VALIDATION_STEPS,
    TEST_STEPS,
) = build_data_loaders(
    cfg.batch_size
)

print(
    "Configured batch size:",
    f"{cfg.batch_size:,}",
)
print(
    "Train samples:",
    f"{len(TRAIN_DATASET):,}",
)
print(
    "Validation samples:",
    f"{len(VALIDATION_DATASET):,}",
)
print(
    "Test samples:",
    f"{len(TEST_DATASET):,}",
)
print(
    "Train steps/epoch:",
    f"{TRAIN_STEPS:,}",
)
print(
    "Validation steps:",
    f"{VALIDATION_STEPS:,}",
)

old_effective_samples = (
    3072 * 1333
)

new_effective_samples = (
    cfg.batch_size
    * TRAIN_STEPS
)

print(
    "Approximate training windows/epoch:",
    f"{new_effective_samples:,}",
)
print(
    "Relative to previous profile:",
    f"{new_effective_samples / old_effective_samples:.3f}x",
)

sample_x_check, sample_y_check = (
    TRAIN_DATASET[0]
)

if sample_y_check.shape[-1] != len(
    TARGETS
):
    raise RuntimeError(
        f"Dataset target width "
        f"{sample_y_check.shape[-1]} "
        f"does not match TARGETS "
        f"length {len(TARGETS)}."
    )

if not torch.isfinite(
    sample_y_check
).all():
    raise RuntimeError(
        "First dataset target sample "
        "contains NaN or infinity."
    )

print(
    "Depth-target dataset validation passed."
)

print(
    "Sample x shape:",
    tuple(sample_x_check.shape),
)

print(
    "Sample y shape:",
    tuple(sample_y_check.shape),
)

## 9. Causal Transformer and 24 vectorized recurrent agents

## Joint initial-training architecture

At each supervised step:

I. the Transformer encodes the current market sequence into the implicit state h_t;

II. all K recurrent agents update their own hidden state using h_t and their own prior action;

III. the learned gate produces population weights;

IV. the weighted aggregate action drives the stochastic transition head;

V. the total loss backpropagates through the transition head, gate, agents, and Transformer together.

Thus this Notebook is intentionally slow. It is the full structural training run, not the later lightweight adaptation stage.

In [ ]:
class Position(nn.Module):
    def __init__(
        self,
        d_model,
        max_len=1024,
    ):
        super().__init__()

        position = (
            torch.arange(max_len)
            .float()
            .unsqueeze(1)
        )

        divisor = torch.exp(
            torch.arange(
                0,
                d_model,
                2,
            ).float()
            * (
                -math.log(10000.0)
                / d_model
            )
        )

        encoding = torch.zeros(
            max_len,
            d_model,
        )

        encoding[:, 0::2] = torch.sin(
            position * divisor
        )

        encoding[:, 1::2] = torch.cos(
            position * divisor
        )

        self.register_buffer(
            "encoding",
            encoding.unsqueeze(0),
            persistent=False,
        )

    def forward(self, x):
        return (
            x
            + self.encoding[
                :, :x.size(1)
            ]
        )


class ConfigurableMarketEncoder(nn.Module):
    """
    Shared market encoder.

    The current experiment uses a causal Transformer, but causality,
    layer count, head count, width, and the optional external channel
    are configuration parameters.
    """

    def __init__(self):
        super().__init__()

        if cfg.encoder_type != "transformer":
            raise ValueError(
                "This Notebook currently implements encoder_type='transformer'. "
                "Other encoder types require a separate implementation."
            )

        self.market_projection = nn.Linear(
            len(FEATURES),
            cfg.d_model,
        )

        self.external_projection = (
            nn.Linear(
                cfg.external_feature_dim,
                cfg.d_model,
            )
            if cfg.external_feature_dim > 0
            else None
        )

        self.source_fusion = (
            nn.Linear(
                2 * cfg.d_model,
                cfg.d_model,
            )
            if cfg.external_feature_dim > 0
            else None
        )

        self.position = Position(
            cfg.d_model
        )

        layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.nhead,
            dim_feedforward=cfg.ff_dim,
            dropout=cfg.dropout,
            batch_first=True,
            norm_first=cfg.norm_first,
            activation="gelu",
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            cfg.layers,
        )

        self.output_projection = nn.Linear(
            cfg.d_model,
            cfg.latent_dim,
        )

    def forward(
        self,
        market_features,
        external_features=None,
    ):
        hidden = self.market_projection(
            market_features
        )

        if self.external_projection is not None:
            if external_features is None:
                external_features = torch.zeros(
                    *market_features.shape[:-1],
                    cfg.external_feature_dim,
                    device=market_features.device,
                    dtype=market_features.dtype,
                )

            external_hidden = (
                self.external_projection(
                    external_features
                )
            )

            hidden = self.source_fusion(
                torch.cat(
                    [
                        hidden,
                        external_hidden,
                    ],
                    dim=-1,
                )
            )

        hidden = self.position(hidden)

        attention_mask = None

        if cfg.encoder_causal:
            sequence_len = (
                market_features.size(1)
            )

            attention_mask = torch.triu(
                torch.ones(
                    sequence_len,
                    sequence_len,
                    dtype=torch.bool,
                    device=market_features.device,
                ),
                diagonal=1,
            )

        hidden = self.transformer(
            hidden,
            mask=attention_mask,
        )

        return self.output_projection(
            hidden
        )


class VectorizedAgentPopulation(nn.Module):
    """
    K independent recurrent agents.

    Agents share the encoder output h_t but do not receive other
    agents' hidden states or actions at the same time step.
    """

    def __init__(self):
        super().__init__()

        self.ids = [
            f"latent_agent_{index}"
            for index in range(
                cfg.num_agents
            )
        ]

        input_dim = (
            cfg.latent_dim
            + cfg.action_dim
        )

        self.weight_ih = nn.Parameter(
            torch.empty(
                cfg.num_agents,
                3 * cfg.agent_hidden,
                input_dim,
            )
        )

        self.weight_hh = nn.Parameter(
            torch.empty(
                cfg.num_agents,
                3 * cfg.agent_hidden,
                cfg.agent_hidden,
            )
        )

        self.bias_ih = nn.Parameter(
            torch.zeros(
                cfg.num_agents,
                3 * cfg.agent_hidden,
            )
        )

        self.bias_hh = nn.Parameter(
            torch.zeros(
                cfg.num_agents,
                3 * cfg.agent_hidden,
            )
        )

        self.body_weight = nn.Parameter(
            torch.empty(
                cfg.num_agents,
                cfg.agent_hidden,
                cfg.agent_hidden,
            )
        )

        self.body_bias = nn.Parameter(
            torch.zeros(
                cfg.num_agents,
                cfg.agent_hidden,
            )
        )

        self.action_mean_weight = nn.Parameter(
            torch.empty(
                cfg.num_agents,
                cfg.action_dim,
                cfg.agent_hidden,
            )
        )

        self.action_mean_bias = nn.Parameter(
            torch.zeros(
                cfg.num_agents,
                cfg.action_dim,
            )
        )

        self.action_log_std_weight = nn.Parameter(
            torch.empty(
                cfg.num_agents,
                cfg.action_dim,
                cfg.agent_hidden,
            )
        )

        self.action_log_std_bias = nn.Parameter(
            torch.zeros(
                cfg.num_agents,
                cfg.action_dim,
            )
        )

        self.group_gate = nn.Sequential(
            nn.Linear(
                cfg.latent_dim,
                cfg.latent_dim,
            ),
            nn.GELU(),
            nn.Linear(
                cfg.latent_dim,
                cfg.num_agents,
            ),
        )

        self.reset_parameters()

    def reset_parameters(self):
        for index in range(
            cfg.num_agents
        ):
            nn.init.xavier_uniform_(
                self.weight_ih[index]
            )

            nn.init.orthogonal_(
                self.weight_hh[index]
            )

            nn.init.xavier_uniform_(
                self.body_weight[index]
            )

            nn.init.xavier_uniform_(
                self.action_mean_weight[
                    index
                ]
            )

            nn.init.xavier_uniform_(
                self.action_log_std_weight[
                    index
                ]
            )

    def initial_state(
        self,
        batch_size,
        device,
    ):
        hidden = torch.zeros(
            batch_size,
            cfg.num_agents,
            cfg.agent_hidden,
            device=device,
        )

        previous_action = torch.zeros(
            batch_size,
            cfg.num_agents,
            cfg.action_dim,
            device=device,
        )

        return hidden, previous_action

    def gru_step(
        self,
        agent_input,
        hidden,
    ):
        input_gates = (
            torch.einsum(
                "bkd,khd->bkh",
                agent_input,
                self.weight_ih,
            )
            + self.bias_ih.unsqueeze(0)
        )

        hidden_gates = (
            torch.einsum(
                "bkd,khd->bkh",
                hidden,
                self.weight_hh,
            )
            + self.bias_hh.unsqueeze(0)
        )

        (
            input_reset,
            input_update,
            input_new,
        ) = input_gates.chunk(
            3,
            dim=-1,
        )

        (
            hidden_reset,
            hidden_update,
            hidden_new,
        ) = hidden_gates.chunk(
            3,
            dim=-1,
        )

        reset_gate = torch.sigmoid(
            input_reset + hidden_reset
        )

        update_gate = torch.sigmoid(
            input_update + hidden_update
        )

        new_gate = torch.tanh(
            input_new
            + reset_gate * hidden_new
        )

        return (
            new_gate
            + update_gate
            * (
                hidden - new_gate
            )
        )

    def forward(
        self,
        market_state,
        hidden,
        previous_action,
        deterministic=False,
    ):
        market_by_agent = (
            market_state
            .unsqueeze(1)
            .expand(
                -1,
                cfg.num_agents,
                -1,
            )
        )

        agent_input = torch.cat(
            [
                market_by_agent,
                previous_action,
            ],
            dim=-1,
        )

        next_hidden = self.gru_step(
            agent_input,
            hidden,
        )

        body = F.gelu(
            torch.einsum(
                "bkd,khd->bkh",
                next_hidden,
                self.body_weight,
            )
            + self.body_bias.unsqueeze(0)
        )

        means = (
            torch.einsum(
                "bkh,kah->bka",
                body,
                self.action_mean_weight,
            )
            + self.action_mean_bias
            .unsqueeze(0)
        )

        log_stds = (
            torch.einsum(
                "bkh,kah->bka",
                body,
                self.action_log_std_weight,
            )
            + self.action_log_std_bias
            .unsqueeze(0)
        ).clamp(-5.0, 2.0)

        actions = (
            means
            if deterministic
            else (
                means
                + log_stds.exp()
                * torch.randn_like(means)
            )
        )

        # The gate observes the shared latent market state.
        # It does not create agent-to-agent communication.
        weights = torch.softmax(
            self.group_gate(
                market_state
            ),
            dim=-1,
        )

        aggregate_action = (
            actions
            * weights.unsqueeze(-1)
        ).sum(dim=1)

        return {
            "actions": actions,
            "means": means,
            "log_stds": log_stds,
            "weights": weights,
            "aggregate_action":
                aggregate_action,
            "hidden": next_hidden,
            "previous": actions,
        }


class MarketTransitionHead(nn.Module):
    """
    Maps the weighted population action to six next-step
    market-transition distributions.

    There is intentionally no direct h_t input.
    """

    def __init__(self):
        super().__init__()

        self.body = nn.Sequential(
            nn.Linear(
                cfg.action_dim,
                192,
            ),
            nn.GELU(),
            nn.Linear(
                192,
                192,
            ),
            nn.GELU(),
        )

        self.state_mean = nn.Linear(
            192,
            cfg.target_dim,
        )

        self.state_log_std = nn.Linear(
            192,
            cfg.target_dim,
        )

    def forward(
        self,
        aggregate_action,
    ):
        hidden = self.body(
            aggregate_action
        )

        return {
            "state_mean":
                self.state_mean(hidden),
            "state_log_std":
                self.state_log_std(hidden)
                .clamp(-5.0, 2.0),
        }


class MarketSimulationModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = (
            ConfigurableMarketEncoder()
        )

        self.population = (
            VectorizedAgentPopulation()
        )

        self.transition = (
            MarketTransitionHead()
        )

    def forward(
        self,
        market_sequence,
        external_sequence=None,
        hidden=None,
        previous_action=None,
        deterministic=False,
    ):
        """
        Advance the encoder and K agent memories through a sequence,
        then predict the transition after the final sequence token.
        """
        token_latents = self.encoder(
            market_sequence,
            external_sequence,
        )

        batch_size = (
            market_sequence.size(0)
        )

        if (
            hidden is None
            or previous_action is None
        ):
            hidden, previous_action = (
                self.population.initial_state(
                    batch_size,
                    market_sequence.device,
                )
            )

        population = None

        for t in range(
            market_sequence.size(1)
        ):
            population = self.population(
                token_latents[:, t],
                hidden,
                previous_action,
                deterministic=deterministic,
            )

            hidden = population["hidden"]
            previous_action = (
                population["previous"]
            )

        output = self.transition(
            population[
                "aggregate_action"
            ]
        )

        return {
            **output,
            "latent_state":
                token_latents[:, -1],
            "actions":
                population["actions"],
            "action_means":
                population["means"],
            "action_log_stds":
                population["log_stds"],
            "weights":
                population["weights"],
            "aggregate_action":
                population[
                    "aggregate_action"
                ],
            "hidden": hidden,
            "previous_action":
                previous_action,
        }

## 10. Build model and inspect parameters

In [ ]:
def parameter_count(module):
    total = sum(
        p.numel()
        for p in module.parameters()
    )

    trainable = sum(
        p.numel()
        for p in module.parameters()
        if p.requires_grad
    )

    return total, trainable


MODEL = MarketSimulationModel().to(
    DEVICE
)

if (
    cfg.use_torch_compile
    and hasattr(torch, "compile")
):
    MODEL = torch.compile(
        MODEL,
        mode=cfg.compile_mode,
    )

print(MODEL)

for name, module in {
    "Encoder": MODEL.encoder,
    "Population": MODEL.population,
    "Transition": MODEL.transition,
    "Full model": MODEL,
}.items():
    total, trainable = (
        parameter_count(module)
    )

    print(
        f"{name:20s} "
        f"total={total/1000:10.3f}K "
        f"trainable={trainable/1000:10.3f}K"
    )

## 11. Sequential current one-second objective

The Transformer processes the whole sequence with a causal mask.

The K independent agent memories are advanced at **every step**. The current experiment uses K=24. Loss is applied only
after the warmup region, which gives the agents enough history before evaluation.

In [ ]:
TARGET_INDEX = {
    name: index
    for index, name in enumerate(TARGETS)
}

RETURN_TARGET_INDEX = TARGET_INDEX["next_return"]


def gaussian_nll(
    target,
    mean,
    log_std,
):
    """
    Numerically stable Gaussian NLL.

    Always evaluate in float32 even when the model forward pass
    runs under float16 or bfloat16 autocast.
    """
    target_f = target.float()
    mean_f = mean.float()
    log_std_f = (
        log_std.float()
        .clamp(-5.0, 2.0)
    )

    squared_error = (
        target_f - mean_f
    ).pow(2)

    inverse_variance = torch.exp(
        -2.0 * log_std_f
    )

    elementwise = (
        0.5
        * squared_error
        * inverse_variance
        + log_std_f
    )

    if not torch.isfinite(
        elementwise
    ).all():
        raise FloatingPointError(
            "Non-finite Gaussian NLL values detected."
        )

    return elementwise.mean()


def diversity_loss(agent_means):
    normalized = F.normalize(agent_means, dim=-1)
    similarity = normalized @ normalized.transpose(-1, -2)
    count = similarity.size(-1)

    mask = (
        1
        - torch.eye(
            count,
            device=similarity.device,
        ).unsqueeze(0)
    )

    return (
        (similarity * mask)
        .pow(2)
        .sum()
        / max(
            1,
            agent_means.size(0)
            * count
            * (count - 1),
        )
    )


def balance_loss(weights):
    uniform = torch.full_like(
        weights,
        1.0 / weights.size(-1),
    )
    return (weights - uniform).pow(2).mean()


def denormalize_target(values, target_index):
    mean = torch.as_tensor(
        TRAIN_DATASET.stats["y_mean"][target_index],
        dtype=values.dtype,
        device=values.device,
    )
    std = torch.as_tensor(
        TRAIN_DATASET.stats["y_std"][target_index],
        dtype=values.dtype,
        device=values.device,
    )
    return values * std + mean


def direction_counts_from_raw_returns(
    actual_return,
    predicted_return,
):
    mask = (
        actual_return.abs()
        > cfg.return_zero_tolerance
    )

    actual_positive = actual_return[mask] > 0
    predicted_positive = predicted_return[mask] > 0

    return {
        "tp": (
            actual_positive
            & predicted_positive
        ).sum().float(),
        "tn": (
            (~actual_positive)
            & (~predicted_positive)
        ).sum().float(),
        "fp": (
            (~actual_positive)
            & predicted_positive
        ).sum().float(),
        "fn": (
            actual_positive
            & (~predicted_positive)
        ).sum().float(),
        "positive_count":
            actual_positive.sum().float(),
        "negative_count":
            (~actual_positive).sum().float(),
    }


def metrics_from_direction_counts(tp, tn, fp, fn):
    eps = 1e-12
    total = tp + tn + fp + fn

    accuracy = (tp + tn) / (total + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    specificity = tn / (tn + fp + eps)

    f1 = (
        2
        * precision
        * recall
        / (
            precision
            + recall
            + eps
        )
    )

    balanced_accuracy = (
        recall + specificity
    ) / 2

    mcc = (
        tp * tn - fp * fn
    ) / torch.sqrt(
        (tp + fp)
        * (tp + fn)
        * (tn + fp)
        * (tn + fn)
        + eps
    )

    return {
        "direction_accuracy_nonzero":
            accuracy,
        "direction_precision":
            precision,
        "direction_recall":
            recall,
        "direction_specificity":
            specificity,
        "direction_f1":
            f1,
        "direction_balanced_accuracy":
            balanced_accuracy,
        "direction_mcc":
            mcc,
    }


def pearson_correlation(
    target,
    prediction,
):
    target_f = target.float()
    prediction_f = prediction.float()

    target_centered = (
        target_f - target_f.mean()
    )

    prediction_centered = (
        prediction_f
        - prediction_f.mean()
    )

    numerator = (
        target_centered
        * prediction_centered
    ).sum()

    denominator = torch.sqrt(
        target_centered
        .pow(2)
        .sum()
        * prediction_centered
        .pow(2)
        .sum()
    )

    if (
        not torch.isfinite(denominator)
        or denominator <= 1e-12
    ):
        return target_f.new_tensor(0.0)

    correlation = (
        numerator / denominator
    )

    return torch.nan_to_num(
        correlation,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )


def sequential_objective(
    model,
    x,
    y,
    deterministic=False,
    return_components=False,
):
    batch_size = x.size(0)
    token_latents = model.encoder(x)

    hidden, previous = (
        model.population.initial_state(
            batch_size,
            x.device,
        )
    )

    state_losses = []
    normalized_maes = []
    normalized_rmses = []
    correlations = []
    diversity_values = []
    balance_values = []

    per_target_maes = [
        []
        for _ in TARGETS
    ]

    tp_total = x.new_tensor(0.0)
    tn_total = x.new_tensor(0.0)
    fp_total = x.new_tensor(0.0)
    fn_total = x.new_tensor(0.0)
    positive_total = x.new_tensor(0.0)
    negative_total = x.new_tensor(0.0)

    for t in range(cfg.sequence_len - 1):
        market_state = token_latents[:, t]

        population = model.population(
            market_state,
            hidden,
            previous,
            deterministic,
        )

        hidden = population["hidden"]
        previous = population["previous"]

        if t < cfg.warmup_len:
            continue

        output = model.transition(
            population["aggregate_action"],
        )

        target_state = y[:, t]
        predicted_state = output["state_mean"]

        state_losses.append(
            gaussian_nll(
                target_state,
                predicted_state,
                output["state_log_std"],
            )
        )

        normalized_error = (
            predicted_state - target_state
        )

        normalized_maes.append(
            normalized_error.abs().mean()
        )

        normalized_rmses.append(
            torch.sqrt(
                normalized_error.pow(2).mean()
                + 1e-12
            )
        )

        actual_return_raw = denormalize_target(
            target_state[:, RETURN_TARGET_INDEX],
            RETURN_TARGET_INDEX,
        )

        predicted_return_raw = denormalize_target(
            predicted_state[:, RETURN_TARGET_INDEX],
            RETURN_TARGET_INDEX,
        )

        correlations.append(
            pearson_correlation(
                actual_return_raw,
                predicted_return_raw,
            )
        )

        counts = direction_counts_from_raw_returns(
            actual_return_raw,
            predicted_return_raw,
        )

        tp_total += counts["tp"]
        tn_total += counts["tn"]
        fp_total += counts["fp"]
        fn_total += counts["fn"]
        positive_total += counts["positive_count"]
        negative_total += counts["negative_count"]

        for target_index in range(len(TARGETS)):
            actual_raw = denormalize_target(
                target_state[:, target_index],
                target_index,
            )

            predicted_raw = denormalize_target(
                predicted_state[:, target_index],
                target_index,
            )

            per_target_maes[target_index].append(
                (
                    predicted_raw - actual_raw
                )
                .abs()
                .mean()
            )

        diversity_values.append(
            diversity_loss(
                population["means"]
            )
        )

        balance_values.append(
            balance_loss(
                population["weights"]
            )
        )

    supervised_step_count = len(
        state_losses
    )

    if (
        supervised_step_count
        != cfg.prediction_len
    ):
        raise RuntimeError(
            f"Expected {cfg.prediction_len} supervised positions, "
            f"but objective produced {supervised_step_count}."
        )

    state_nll = torch.stack(
        state_losses
    ).mean()

    state_mae = torch.stack(
        normalized_maes
    ).mean()

    state_rmse = torch.stack(
        normalized_rmses
    ).mean()

    return_correlation = torch.stack(
        correlations
    ).mean()

    diversity = torch.stack(
        diversity_values
    ).mean()

    balance = torch.stack(
        balance_values
    ).mean()

    total = (
        cfg.lambda_state * state_nll
        + cfg.lambda_diversity * diversity
        + cfg.lambda_balance * balance
    )

    result = {
        "total": total,
        "state_nll": state_nll,
        "state_mae_normalized":
            state_mae,
        "state_rmse_normalized":
            state_rmse,
        "return_correlation":
            return_correlation,
        "direction_majority_baseline":
            torch.maximum(
                positive_total,
                negative_total,
            )
            / (
                positive_total
                + negative_total
                + 1e-12
            ),
        "direction_positive_rate":
            positive_total
            / (
                positive_total
                + negative_total
                + 1e-12
            ),
        "diversity": diversity,
        "balance": balance,
    }

    result.update(
        metrics_from_direction_counts(
            tp_total,
            tn_total,
            fp_total,
            fn_total,
        )
    )

    for target_index, target_name in enumerate(
        TARGETS
    ):
        result[
            f"mae_raw_{target_name}"
        ] = torch.stack(
            per_target_maes[
                target_index
            ]
        ).mean()

    if return_components:
        return result

    return total

## 12. Scheduler definition

In [ ]:
def build_scheduler(
    optimizer,
    steps_per_epoch,
):
    name = (
        cfg.scheduler_name
        .lower()
        .strip()
    )

    if name in {
        "none",
        "off",
        "disabled",
    }:
        return None, "none"

    if name == "reduce_on_plateau":
        return (
            torch.optim.lr_scheduler
            .ReduceLROnPlateau(
                optimizer,
                mode="min",
                factor=cfg.scheduler_factor,
                patience=cfg.scheduler_patience,
                min_lr=cfg.scheduler_min_lr,
            ),
            "epoch_metric",
        )

    if name == "cosine":
        return (
            torch.optim.lr_scheduler
            .CosineAnnealingLR(
                optimizer,
                T_max=cfg.cosine_t_max,
                eta_min=cfg.cosine_eta_min,
            ),
            "epoch",
        )

    if name == "step":
        return (
            torch.optim.lr_scheduler
            .StepLR(
                optimizer,
                step_size=cfg.step_size,
                gamma=cfg.step_gamma,
            ),
            "epoch",
        )

    if name == "onecycle":
        return (
            torch.optim.lr_scheduler
            .OneCycleLR(
                optimizer,
                max_lr=cfg.lr,
                epochs=cfg.max_epochs,
                steps_per_epoch=steps_per_epoch,
                pct_start=cfg.onecycle_pct_start,
                div_factor=cfg.onecycle_div_factor,
                final_div_factor=cfg.onecycle_final_div_factor,
            ),
            "batch",
        )

    raise ValueError(
        f"Unsupported scheduler: "
        f"{cfg.scheduler_name}"
    )

## 13. Evaluation function

In [ ]:
@torch.no_grad()
def evaluate_loader(
    model,
    loader,
    max_batches,
    description,
):
    model.eval()

    metric_names = [
        "total",
        "state_nll",
        "state_mae_normalized",
        "state_rmse_normalized",
        "return_correlation",
        "direction_accuracy_nonzero",
        "direction_precision",
        "direction_recall",
        "direction_specificity",
        "direction_f1",
        "direction_balanced_accuracy",
        "direction_mcc",
        "direction_majority_baseline",
        "direction_positive_rate",
    ] + [
        f"mae_raw_{name}"
        for name in TARGETS
    ]

    metric_lists = {
        name: []
        for name in metric_names
    }

    progress = tqdm(
        enumerate(loader, start=1),
        total=max_batches,
        desc=description,
        dynamic_ncols=True,
        leave=False,
    )

    for batch_index, (x, y) in progress:
        if batch_index > max_batches:
            break

        x = x.to(
            DEVICE,
            non_blocking=True,
        )

        y = y.to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=AMP_DTYPE,
            enabled=(
                cfg.use_amp
                and DEVICE.type == "cuda"
            ),
        ):
            components = sequential_objective(
                model,
                x,
                y,
                deterministic=True,
                return_components=True,
            )

        for key in metric_lists:
            metric_lists[key].append(
                float(
                    components[key]
                    .detach()
                )
            )

        if (
            batch_index
            % cfg.progress_update_every
            == 0
        ):
            progress.set_postfix(
                loss=(
                    f"{np.mean(metric_lists['total']):.5f}"
                ),
                bal_acc=(
                    f"{np.mean(metric_lists['direction_balanced_accuracy']):.2%}"
                ),
                mcc=(
                    f"{np.mean(metric_lists['direction_mcc']):.3f}"
                ),
                corr=(
                    f"{np.mean(metric_lists['return_correlation']):.3f}"
                ),
            )

    return {
        key: float(np.mean(values))
        for key, values
        in metric_lists.items()
    }

## 14. One-click training

In [ ]:
# Reduced-state reconstruction for recursive stochastic simulation

TARGET_INDEX = {name: index for index, name in enumerate(TARGETS)}
FEATURE_INDEX = {name: index for index, name in enumerate(FEATURES)}


def denormalize_prediction(normalized_prediction):
    y_mean = torch.as_tensor(
        TRAIN_DATASET.stats["y_mean"],
        dtype=normalized_prediction.dtype,
        device=normalized_prediction.device,
    )
    y_std = torch.as_tensor(
        TRAIN_DATASET.stats["y_std"],
        dtype=normalized_prediction.dtype,
        device=normalized_prediction.device,
    )
    return normalized_prediction * y_std + y_mean


def sample_transition_distribution(state_mean, state_log_std, temperature=None, generator=None):
    if temperature is None:
        temperature = cfg.stochastic_temperature
    noise = torch.randn(
        state_mean.shape,
        dtype=state_mean.dtype,
        device=state_mean.device,
        generator=generator,
    )
    return state_mean + temperature * torch.exp(state_log_std) * noise


def reconstruct_reduced_market_state(current_state, raw_transition):
    r = raw_transition[..., TARGET_INDEX["next_return"]]
    ds = raw_transition[..., TARGET_INDEX["next_spread_change"]]
    di = raw_transition[..., TARGET_INDEX["next_imbalance_change"]]
    dd = raw_transition[..., TARGET_INDEX["next_log_total_depth_change"]]
    lts = raw_transition[..., TARGET_INDEX["next_log_trade_size"]]
    ltc = raw_transition[..., TARGET_INDEX["next_log_trade_count"]]

    mid = (current_state["mid_price"] * torch.exp(r)).clamp_min(1e-12)
    spread = (current_state["spread"] + ds).clamp_min(cfg.minimum_spread)
    imbalance = (current_state["imbalance_1"] + di).clamp(-1.0, 1.0)

    log_depth = torch.log1p(current_state["total_depth_1"].clamp_min(0.0)) + dd
    depth = torch.expm1(log_depth).clamp_min(cfg.minimum_total_depth)

    bid_size = (0.5 * depth * (1.0 + imbalance)).clamp_min(0.0)
    ask_size = (0.5 * depth * (1.0 - imbalance)).clamp_min(0.0)
    bid = (mid - 0.5 * spread).clamp_min(1e-12)
    ask = (mid + 0.5 * spread).clamp_min(bid + cfg.minimum_spread)

    trade_size = torch.expm1(lts).clamp_min(0.0)
    trade_count_expected = torch.expm1(ltc).clamp_min(0.0)
    trade_count = torch.round(trade_count_expected)

    return {
        "mid_price": mid,
        "spread": spread,
        "bid_price1": bid,
        "ask_price1": ask,
        "imbalance_1": imbalance,
        "total_depth_1": depth,
        "bid_size1": bid_size,
        "ask_size1": ask_size,
        "trade_size": trade_size,
        "trade_count": trade_count,
        "trade_count_expected": trade_count_expected,
        "return_1": r,
        "seconds_in_bucket": current_state["seconds_in_bucket"] + 1,
        "real_book_update": torch.zeros_like(mid),
        "seconds_since_last_real_update": current_state["seconds_since_last_real_update"] + 1,
    }


def reduced_state_to_feature_tensor(state, stock_id_value):
    mid = state["mid_price"]
    values = {
        "stock_id": torch.full_like(mid, float(stock_id_value)),
        "seconds_in_bucket": state["seconds_in_bucket"],
        "real_book_update": state["real_book_update"],
        "seconds_since_last_real_update": state["seconds_since_last_real_update"],
        "bid_price1": state["bid_price1"],
        "ask_price1": state["ask_price1"],
        "bid_size1": state["bid_size1"],
        "ask_size1": state["ask_size1"],
        "mid_price": state["mid_price"],
        "spread": state["spread"],
        "imbalance_1": state["imbalance_1"],
        "log_total_depth_1": torch.log1p(state["total_depth_1"].clamp_min(0.0)),
        "trade_size": state["trade_size"],
        "trade_count": state["trade_count"],
        "return_1": state["return_1"],
    }
    missing = [name for name in FEATURES if name not in values]
    if missing:
        raise KeyError("Recursive state cannot reconstruct FEATURES: " + ", ".join(missing))
    raw = torch.stack([values[name] for name in FEATURES], dim=-1)
    x_mean = torch.as_tensor(TRAIN_DATASET.stats["x_mean"], dtype=raw.dtype, device=raw.device)
    x_std = torch.as_tensor(TRAIN_DATASET.stats["x_std"], dtype=raw.dtype, device=raw.device)
    return (raw - x_mean) / x_std


@torch.no_grad()
def stochastic_recursive_rollout(model, initial_sequence, initial_state, stock_id_value, steps, temperature=None, seed=1234):
    model.eval()
    generator = torch.Generator(device=initial_sequence.device)
    generator.manual_seed(seed)
    sequence = initial_sequence.clone()
    state = {key: value.clone() for key, value in initial_state.items()}
    states, transitions = [], []

    for _ in range(steps):
        output = model(sequence, deterministic=True)
        normalized_transition = sample_transition_distribution(
            output["state_mean"], output["state_log_std"], temperature, generator
        )
        raw_transition = denormalize_prediction(normalized_transition)
        state = reconstruct_reduced_market_state(state, raw_transition)
        next_row = reduced_state_to_feature_tensor(state, stock_id_value)
        sequence = torch.cat([sequence[:, 1:], next_row.unsqueeze(1)], dim=1)
        transitions.append(raw_transition)
        states.append({key: value.clone() for key, value in state.items()})

    return {"states": states, "transitions": transitions, "final_sequence": sequence}

In [ ]:
def probe_training_batch(
    batch_size=None,
):
    """
    Run one real forward/backward pass without taking an optimizer step.

    This verifies that the selected batch fits and reports peak allocated
    and reserved CUDA memory. Run this before a fresh long training job.
    """
    if batch_size is None:
        batch_size = cfg.batch_size

    (
        probe_loader,
        _,
        _,
        _,
        _,
        _,
    ) = build_data_loaders(
        batch_size
    )

    x, y = next(
        iter(probe_loader)
    )

    x = x.to(
        DEVICE,
        non_blocking=True,
    )

    y = y.to(
        DEVICE,
        non_blocking=True,
    )

    MODEL.train()
    MODEL.zero_grad(
        set_to_none=True
    )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    try:
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=AMP_DTYPE,
            enabled=(
                cfg.use_amp
                and DEVICE.type == "cuda"
            ),
        ):
            components = sequential_objective(
                MODEL,
                x,
                y,
                return_components=True,
            )

        components["total"].backward()

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

            peak_allocated = (
                torch.cuda
                .max_memory_allocated()
                / 1024**3
            )

            peak_reserved = (
                torch.cuda
                .max_memory_reserved()
                / 1024**3
            )

            total_memory = (
                torch.cuda
                .get_device_properties(0)
                .total_memory
                / 1024**3
            )

            print(
                f"Batch {batch_size:,} fits."
            )
            print(
                "Peak allocated:",
                f"{peak_allocated:.2f} GB",
            )
            print(
                "Peak reserved:",
                f"{peak_reserved:.2f} GB",
            )
            print(
                "Physical GPU memory:",
                f"{total_memory:.2f} GB",
            )
            print(
                "Reserved headroom:",
                f"{total_memory - peak_reserved:.2f} GB",
            )

        return True

    except torch.cuda.OutOfMemoryError:
        print(
            f"Batch {batch_size:,} caused CUDA OOM."
        )
        return False

    finally:
        MODEL.zero_grad(
            set_to_none=True
        )

        del x, y

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()


def choose_largest_safe_batch():
    """
    Probe the configured descending fallback list.
    The first fitting size becomes cfg.batch_size and rebuilds all loaders.
    """
    global TRAIN_LOADER
    global VALIDATION_LOADER
    global TEST_LOADER
    global TRAIN_STEPS
    global VALIDATION_STEPS
    global TEST_STEPS

    for candidate in (
        cfg.aggressive_batch_fallback
    ):
        print(
            "\nProbing batch size:",
            f"{candidate:,}",
        )

        if probe_training_batch(
            candidate
        ):
            cfg.batch_size = candidate

            (
                TRAIN_LOADER,
                VALIDATION_LOADER,
                TEST_LOADER,
                TRAIN_STEPS,
                VALIDATION_STEPS,
                TEST_STEPS,
            ) = build_data_loaders(
                candidate
            )

            print(
                "\nSelected batch size:",
                f"{candidate:,}",
            )

            return candidate

    raise RuntimeError(
        "None of the aggressive batch sizes fit."
    )

# Recommended before training on a new GPU/runtime:
#
# SELECTED_BATCH_SIZE = choose_largest_safe_batch()

In [ ]:
def assert_finite_training_state():
    """
    Validate one full batch and one model objective before training.
    """
    print("AMP dtype:", AMP_DTYPE)
    print("TARGETS:", TARGETS)
    print("y_mean:", TRAIN_DATASET.stats["y_mean"])
    print("y_std:", TRAIN_DATASET.stats["y_std"])

    for key, values in (
        TRAIN_DATASET.stats.items()
    ):
        values = np.asarray(values)

        if not np.isfinite(values).all():
            raise RuntimeError(
                f"Non-finite normalization statistic: {key}"
            )

    x, y = next(
        iter(TRAIN_LOADER)
    )

    if not torch.isfinite(x).all():
        raise RuntimeError(
            "Input batch contains NaN or infinity."
        )

    if not torch.isfinite(y).all():
        raise RuntimeError(
            "Target batch contains NaN or infinity."
        )

    x = x.to(
        DEVICE,
        non_blocking=True,
    )

    y = y.to(
        DEVICE,
        non_blocking=True,
    )

    MODEL.train()
    MODEL.zero_grad(
        set_to_none=True
    )

    with torch.autocast(
        device_type=DEVICE.type,
        dtype=AMP_DTYPE,
        enabled=(
            cfg.use_amp
            and DEVICE.type == "cuda"
        ),
    ):
        components = sequential_objective(
            MODEL,
            x,
            y,
            return_components=True,
        )

    for key, value in components.items():
        if torch.is_tensor(value):
            finite = torch.isfinite(
                value
            ).all().item()

            print(
                key,
                float(
                    value.detach()
                    .float()
                    .cpu()
                ),
                "finite=",
                finite,
            )

            if not finite:
                raise RuntimeError(
                    f"Preflight metric is non-finite: {key}"
                )

    components["total"].backward()

    finite_gradients = True

    for name, parameter in (
        MODEL.named_parameters()
    ):
        if (
            parameter.grad is not None
            and not torch.isfinite(
                parameter.grad
            ).all()
        ):
            print(
                "Non-finite gradient:",
                name,
            )
            finite_gradients = False
            break

    MODEL.zero_grad(
        set_to_none=True
    )

    del x, y, components

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    if not finite_gradients:
        raise RuntimeError(
            "Preflight backward pass produced non-finite gradients."
        )

    print(
        "Finite training preflight passed."
    )


assert_finite_training_state()

In [ ]:
def current_lr(optimizer):
    return optimizer.param_groups[0]["lr"]


BEST_CHECKPOINT_PATH = (
    DRIVE_CHECKPOINT_ROOT
    / (
        f"optiver_{cfg.decision_interval_seconds:g}s_"
        f"{'causal' if cfg.encoder_causal else 'noncausal'}_"
        f"K{cfg.num_agents}_best.pt"
    )
)

LATEST_CHECKPOINT_PATH = (
    DRIVE_CHECKPOINT_ROOT
    / (
        f"optiver_{cfg.decision_interval_seconds:g}s_"
        f"{'causal' if cfg.encoder_causal else 'noncausal'}_"
        f"K{cfg.num_agents}_latest.pt"
    )
)

RESUME_TRAINING = False


def unwrap_model(model):
    return (
        model._orig_mod
        if hasattr(model, "_orig_mod")
        else model
    )


def clone_state_dict_to_cpu(model):
    raw_model = unwrap_model(model)

    return {
        key: value.detach().cpu().clone()
        for key, value
        in raw_model.state_dict().items()
    }


def atomic_torch_save(
    payload,
    destination,
):
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = (
        destination.parent
        / f"{destination.name}.tmp"
    )

    torch.save(
        payload,
        temporary_path,
    )

    temporary_path.replace(
        destination
    )


def optimizer_to_device(
    optimizer,
    device,
):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)


def build_training_checkpoint(
    *,
    epoch,
    model,
    optimizer,
    scheduler,
    scheduler_mode,
    scaler,
    best_validation,
    bad_epochs,
    history,
    best_state,
    interrupted=False,
):
    raw_model = unwrap_model(model)

    return {
        "schema_version": "3.2.0",
        "dataset":
            "optiver-realized-volatility-prediction",
        "frequency":
            "one-second resampled grid",
        "epoch": int(epoch),
        "interrupted": bool(interrupted),
        "model_state_dict":
            clone_state_dict_to_cpu(raw_model),
        "optimizer_state_dict":
            optimizer.state_dict(),
        "scheduler_state_dict": (
            scheduler.state_dict()
            if scheduler is not None
            else None
        ),
        "scheduler_mode": scheduler_mode,
        "scaler_state_dict":
            scaler.state_dict(),
        "best_validation":
            float(best_validation),
        "bad_epochs":
            int(bad_epochs),
        "history":
            list(history),
        "best_state":
            best_state,
        "config":
            asdict(cfg),
        "features":
            FEATURES,
        "targets":
            TARGETS,
        "normalization":
            TRAIN_DATASET.stats,
        "agent_ids":
            list(raw_model.population.ids),
        "architecture": {
            "causal_transformer": True,
            "agent_recurrence_each_second": True,
            "agent_count":
                cfg.num_agents,
            "next_event_time_head": False,
            "next_second_update_head": False,
        },
    }


def load_training_checkpoint(
    *,
    checkpoint_path,
    model,
    optimizer,
    scheduler,
    scaler,
):
    checkpoint_path = Path(
        checkpoint_path
    )

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Checkpoint does not exist: "
            f"{checkpoint_path}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    raw_model = unwrap_model(model)

    raw_model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    optimizer.load_state_dict(
        checkpoint[
            "optimizer_state_dict"
        ]
    )

    optimizer_to_device(
        optimizer,
        DEVICE,
    )

    saved_scheduler_state = (
        checkpoint.get(
            "scheduler_state_dict"
        )
    )

    if (
        scheduler is not None
        and saved_scheduler_state
        is not None
    ):
        scheduler.load_state_dict(
            saved_scheduler_state
        )

    saved_scaler_state = (
        checkpoint.get(
            "scaler_state_dict"
        )
    )

    if saved_scaler_state:
        scaler.load_state_dict(
            saved_scaler_state
        )

    return (
        int(
            checkpoint.get(
                "epoch",
                0,
            )
        ) + 1,
        float(
            checkpoint.get(
                "best_validation",
                float("inf"),
            )
        ),
        int(
            checkpoint.get(
                "bad_epochs",
                0,
            )
        ),
        list(
            checkpoint.get(
                "history",
                [],
            )
        ),
        checkpoint.get(
            "best_state"
        ),
    )


def train_model(
    resume_training=RESUME_TRAINING,
):
    optimizer = torch.optim.AdamW(
        MODEL.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
        fused=(
            DEVICE.type == "cuda"
        ),
    )

    scheduler, scheduler_mode = (
        build_scheduler(
            optimizer,
            steps_per_epoch=TRAIN_STEPS,
        )
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(
            cfg.use_amp
            and DEVICE.type == "cuda"
            and AMP_DTYPE
            == torch.float16
        )
    )

    start_epoch = 1
    best_validation = float("inf")
    best_state = None
    bad_epochs = 0
    history = []

    if (
        resume_training
        and LATEST_CHECKPOINT_PATH.exists()
    ):
        (
            start_epoch,
            best_validation,
            bad_epochs,
            history,
            best_state,
        ) = load_training_checkpoint(
            checkpoint_path=
                LATEST_CHECKPOINT_PATH,
            model=MODEL,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
        )

        print(
            "Resuming from epoch:",
            start_epoch,
        )

    print("=" * 96)
    print("TRAINING START")
    print("=" * 96)

    current_epoch = (
        start_epoch - 1
    )

    try:
        for epoch in range(
            start_epoch,
            cfg.max_epochs + 1,
        ):
            current_epoch = epoch
            MODEL.train()
            epoch_start = time.time()

            optimizer.zero_grad(
                set_to_none=True
            )

            running = {
                "total": [],
                "state_nll": [],
                "state_mae_normalized": [],
                "state_rmse_normalized": [],
                "return_correlation": [],
                "direction_accuracy_nonzero": [],
                "direction_precision": [],
                "direction_recall": [],
                "direction_specificity": [],
                "direction_f1": [],
                "direction_balanced_accuracy": [],
                "direction_mcc": [],
                "direction_majority_baseline": [],
                "direction_positive_rate": [],
            }

            for target_name in TARGETS:
                running[
                    f"mae_raw_{target_name}"
                ] = []

            progress = tqdm(
                enumerate(
                    TRAIN_LOADER,
                    start=1,
                ),
                total=TRAIN_STEPS,
                desc=(
                    f"Epoch "
                    f"{epoch}/"
                    f"{cfg.max_epochs}"
                ),
                dynamic_ncols=True,
                leave=True,
            )

            for batch_index, (
                x,
                y,
            ) in progress:
                if (
                    batch_index
                    > TRAIN_STEPS
                ):
                    break

                x = x.to(
                    DEVICE,
                    non_blocking=True,
                )

                y = y.to(
                    DEVICE,
                    non_blocking=True,
                )

                with torch.autocast(
                    device_type=
                        DEVICE.type,
                    dtype=AMP_DTYPE,
                    enabled=(
                        cfg.use_amp
                        and DEVICE.type
                        == "cuda"
                    ),
                ):
                    components = (
                        sequential_objective(
                            MODEL,
                            x,
                            y,
                            return_components=True,
                        )
                    )

                    loss = (
                        components["total"]
                        / cfg
                        .gradient_accumulation_steps
                    )

                if not torch.isfinite(loss):
                    raise FloatingPointError(
                        "Training loss became non-finite "
                        f"at epoch={epoch}, batch={batch_index}."
                    )

                scaler.scale(
                    loss
                ).backward()

                should_step = (
                    batch_index
                    % cfg
                    .gradient_accumulation_steps
                    == 0
                    or batch_index
                    == TRAIN_STEPS
                )

                if should_step:
                    scaler.unscale_(
                        optimizer
                    )

                    gradient_norm = (
                        torch.nn.utils
                        .clip_grad_norm_(
                            MODEL.parameters(),
                            cfg.grad_clip,
                            error_if_nonfinite=True,
                        )
                    )

                    scaler.step(
                        optimizer
                    )

                    scaler.update()

                    optimizer.zero_grad(
                        set_to_none=True
                    )

                    if (
                        scheduler
                        is not None
                        and scheduler_mode
                        == "batch"
                    ):
                        scheduler.step()

                for key in running:
                    running[key].append(
                        float(
                            components[key]
                            .detach()
                        )
                    )

                if (
                    batch_index
                    % cfg
                    .progress_update_every
                    == 0
                ):
                    allocated = (
                        torch.cuda
                        .memory_allocated()
                        / 1024**3
                        if DEVICE.type
                        == "cuda"
                        else 0.0
                    )

                    progress.set_postfix(
                        loss=(
                            f"{np.mean(running['total']):.5f}"
                        ),
                        mcc=(
                            f"{np.mean(running['direction_mcc']):.3f}"
                        ),
                        bal_acc=(
                            f"{np.mean(running['direction_balanced_accuracy']):.2%}"
                        ),
                        corr=(
                            f"{np.mean(running['return_correlation']):.3f}"
                        ),
                        lr=(
                            f"{current_lr(optimizer):.2e}"
                        ),
                        gpu=(
                            f"{allocated:.1f}GB"
                        ),
                    )

            train_metrics = {
                key:
                    float(
                        np.mean(values)
                    )
                for key, values
                in running.items()
            }

            validation_metrics = (
                evaluate_loader(
                    MODEL,
                    VALIDATION_LOADER,
                    VALIDATION_STEPS,
                    (
                        f"Validation "
                        f"{epoch}/"
                        f"{cfg.max_epochs}"
                    ),
                )
            )

            if (
                scheduler is not None
                and scheduler_mode
                == "epoch_metric"
            ):
                scheduler.step(
                    validation_metrics[
                        "total"
                    ]
                )

            elif (
                scheduler is not None
                and scheduler_mode
                == "epoch"
            ):
                scheduler.step()

            improved = (
                validation_metrics[
                    "total"
                ]
                < best_validation
                - cfg.min_delta
            )

            if improved:
                best_validation = float(
                    validation_metrics[
                        "total"
                    ]
                )

                bad_epochs = 0

                best_state = (
                    clone_state_dict_to_cpu(
                        MODEL
                    )
                )

                status = "BEST"

            else:
                bad_epochs += 1

                status = (
                    "no improvement "
                    f"({bad_epochs}/"
                    f"{cfg.patience})"
                )

            duration = (
                time.time()
                - epoch_start
            )

            record = {
                "epoch": epoch,
                "learning_rate":
                    current_lr(
                        optimizer
                    ),
                "duration_seconds":
                    duration,
                "status":
                    status,
            }

            for key, value in (
                train_metrics.items()
            ):
                record[
                    f"train_{key}"
                ] = value

            for key, value in (
                validation_metrics
                .items()
            ):
                record[
                    f"validation_{key}"
                ] = value

            history.append(
                record
            )

            latest_checkpoint = (
                build_training_checkpoint(
                    epoch=epoch,
                    model=MODEL,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    scheduler_mode=
                        scheduler_mode,
                    scaler=scaler,
                    best_validation=
                        best_validation,
                    bad_epochs=
                        bad_epochs,
                    history=history,
                    best_state=
                        best_state,
                    interrupted=False,
                )
            )

            atomic_torch_save(
                latest_checkpoint,
                LATEST_CHECKPOINT_PATH,
            )

            if improved:
                best_checkpoint = (
                    dict(
                        latest_checkpoint
                    )
                )

                best_checkpoint[
                    "model_state_dict"
                ] = {
                    key:
                        value.clone()
                    for key, value
                    in best_state.items()
                }

                atomic_torch_save(
                    best_checkpoint,
                    BEST_CHECKPOINT_PATH,
                )

            print(
                f"\nEpoch {epoch}/{cfg.max_epochs}\n"
                f"  train loss: {train_metrics['total']:.6f}\n"
                f"  validation loss: {validation_metrics['total']:.6f}\n"
                f"  normalized MAE / RMSE: "
                f"{validation_metrics['state_mae_normalized']:.6f} / "
                f"{validation_metrics['state_rmse_normalized']:.6f}\n"
                f"  return correlation: "
                f"{validation_metrics['return_correlation']:.4f}\n"
                f"  nonzero direction accuracy: "
                f"{validation_metrics['direction_accuracy_nonzero']:.2%}\n"
                f"  balanced accuracy / MCC: "
                f"{validation_metrics['direction_balanced_accuracy']:.2%} / "
                f"{validation_metrics['direction_mcc']:.4f}\n"
                f"  precision / recall / F1: "
                f"{validation_metrics['direction_precision']:.2%} / "
                f"{validation_metrics['direction_recall']:.2%} / "
                f"{validation_metrics['direction_f1']:.2%}\n"
                f"  positive rate / majority baseline: "
                f"{validation_metrics['direction_positive_rate']:.2%} / "
                f"{validation_metrics['direction_majority_baseline']:.2%}\n"
                f"  raw MAE next return: "
                f"{validation_metrics['mae_raw_next_return']:.8f}\n"
                f"  raw MAE spread change: "
                f"{validation_metrics['mae_raw_next_spread_change']:.8f}\n"
                f"  raw MAE imbalance change: "
                f"{validation_metrics['mae_raw_next_imbalance_change']:.6f}\n"
                f"  raw MAE log depth change: "
                f"{validation_metrics['mae_raw_next_log_total_depth_change']:.6f}\n"
                f"  raw MAE log trade size/count: "
                f"{validation_metrics['mae_raw_next_log_trade_size']:.6f} / "
                f"{validation_metrics['mae_raw_next_log_trade_count']:.6f}\n"
                f"  learning rate: {current_lr(optimizer):.3e}\n"
                f"  duration: {duration/60:.2f} minutes\n"
                f"  status: {status}\n"
            )

            if (
                bad_epochs
                >= cfg.patience
            ):
                print(
                    "Early stopping."
                )
                break

    except KeyboardInterrupt:
        print(
            "\nTraining interrupted manually."
        )

        emergency_checkpoint = (
            build_training_checkpoint(
                epoch=current_epoch,
                model=MODEL,
                optimizer=optimizer,
                scheduler=scheduler,
                scheduler_mode=
                    scheduler_mode,
                scaler=scaler,
                best_validation=
                    best_validation,
                bad_epochs=
                    bad_epochs,
                history=history,
                best_state=
                    best_state,
                interrupted=True,
            )
        )

        atomic_torch_save(
            emergency_checkpoint,
            LATEST_CHECKPOINT_PATH,
        )

        print(
            "Emergency checkpoint saved:",
            LATEST_CHECKPOINT_PATH,
        )

    if best_state is None:
        raise RuntimeError(
            "No best checkpoint was produced."
        )

    raw_model = unwrap_model(
        MODEL
    )

    raw_model.load_state_dict(
        best_state
    )

    final_checkpoint = {
        "schema_version":
            "3.2.0",
        "dataset":
            "optiver-realized-volatility-prediction",
        "frequency":
            "one-second resampled grid",
        "config":
            asdict(cfg),
        "features":
            FEATURES,
        "targets":
            TARGETS,
        "normalization":
            TRAIN_DATASET.stats,
        "agent_ids":
            list(
                raw_model.population.ids
            ),
        "model":
            clone_state_dict_to_cpu(
                raw_model
            ),
        "best_validation":
            float(
                best_validation
            ),
        "history":
            history,
        "architecture": {
            "encoder_type":
                cfg.encoder_type,
            "encoder_causal":
                cfg.encoder_causal,
            "decision_interval_seconds":
                cfg.decision_interval_seconds,
            "external_feature_dim":
                cfg.external_feature_dim,
            "agent_count":
                cfg.num_agents,
            "agent_communication":
                "none_within_step",
            "transition_input":
                "weighted_aggregate_action_only",
            "agent_recurrence_each_second":
                True,
            "agent_count":
                cfg.num_agents,
            "next_event_time_head":
                False,
            "next_second_update_head":
                False,
        },
    }

    return (
        MODEL,
        final_checkpoint,
    )

## 15. Start training

In [ ]:
# Remove checkpoints produced by the NaN run before starting fresh.
for contaminated_path in [
    LATEST_CHECKPOINT_PATH,
    BEST_CHECKPOINT_PATH,
]:
    if contaminated_path.exists():
        contaminated_path.unlink()
        print(
            "Removed contaminated checkpoint:",
            contaminated_path,
        )

RESUME_TRAINING = False

In [ ]:
TRAINED_MODEL, CHECKPOINT = train_model()

## 16. Held-out test evaluation

In [ ]:
TEST_METRICS = evaluate_loader(
    TRAINED_MODEL,
    TEST_LOADER,
    TEST_STEPS,
    "Held-out test",
)

print("=" * 96)
print("TEST RESULTS")
print("=" * 96)

for key, value in TEST_METRICS.items():
    if "accuracy" in key or "precision" in key or "recall" in key or "f1" in key or "baseline" in key:
        print(
            f"{key:32s}: {value:.2%}"
        )
    else:
        print(
            f"{key:32s}: {value:.6f}"
        )

CHECKPOINT[
    "test_metrics"
] = TEST_METRICS

## 17. Save checkpoint

In [ ]:
LOCAL_CHECKPOINT_PATH = Path(
    "/content"
) / (
    f"optiver_{cfg.decision_interval_seconds:g}s_"
    f"{'causal' if cfg.encoder_causal else 'noncausal'}_"
    f"K{cfg.num_agents}.pt"
)

DRIVE_CHECKPOINT_PATH = (
    DRIVE_CHECKPOINT_ROOT
    / (
        f"optiver_{cfg.decision_interval_seconds:g}s_"
        f"{'causal' if cfg.encoder_causal else 'noncausal'}_"
        f"K{cfg.num_agents}.pt"
    )
)

torch.save(
    CHECKPOINT,
    LOCAL_CHECKPOINT_PATH,
)

torch.save(
    CHECKPOINT,
    DRIVE_CHECKPOINT_PATH,
)

print("Local:", LOCAL_CHECKPOINT_PATH)
print("Drive:", DRIVE_CHECKPOINT_PATH)

## 18. Training history

In [ ]:
HISTORY = pd.DataFrame(
    CHECKPOINT["history"]
)

display(HISTORY)

plt.figure(figsize=(10, 5))
plt.plot(
    HISTORY["epoch"],
    HISTORY["train_total"],
    marker="o",
    label="Train",
)
plt.plot(
    HISTORY["epoch"],
    HISTORY["validation_total"],
    marker="o",
    label="Validation",
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training objective")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(
    HISTORY["epoch"],
    HISTORY["validation_direction_f1"],
    marker="o",
    label="Direction F1",
)
plt.plot(
    HISTORY["epoch"],
    HISTORY[
        "validation_direction_balanced_accuracy"
    ],
    marker="o",
    label="Balanced accuracy",
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 19. Runtime recurrent-state contract

Maintain one canonical real hidden state per stock.

For simulation:

1. clone the canonical real hidden state;
2. generate one-second predicted states recursively;
3. update only the cloned branch hidden state;
4. when a new real observation arrives, discard/freeze the simulated branch;
5. update the canonical state from the real one-second sequence;
6. start a new forecast branch.

Simulated GRU memory must never overwrite canonical real GRU memory.

## Export structural checkpoints

The initial run exports one complete simulator checkpoint and separate module checkpoints. The separate participant-side checkpoint can later be used by daily fine-tuning and runtime adaptation files without retraining the Transformer every time.

In [ ]:
FULL_MODEL_CHECKPOINT_PATH = Path(
    "/content"
) / (
    f"full_market_simulator_"
    f"{cfg.decision_interval_seconds:g}s_"
    f"{'causal' if cfg.encoder_causal else 'noncausal'}_"
    f"K{cfg.num_agents}.pt"
)

ENCODER_CHECKPOINT_PATH = Path(
    "/content"
) / (
    f"market_encoder_"
    f"{cfg.decision_interval_seconds:g}s_"
    f"{'causal' if cfg.encoder_causal else 'noncausal'}_"
    f"L{cfg.layers}_H{cfg.nhead}_D{cfg.latent_dim}.pt"
)

PARTICIPANT_CHECKPOINT_PATH = Path(
    "/content"
) / (
    f"participant_population_"
    f"{cfg.decision_interval_seconds:g}s_"
    f"K{cfg.num_agents}_"
    f"H{cfg.agent_hidden}_A{cfg.action_dim}.pt"
)

TRANSITION_CHECKPOINT_PATH = Path(
    "/content"
) / (
    f"market_transition_"
    f"{cfg.decision_interval_seconds:g}s_"
    f"K{cfg.num_agents}.pt"
)

raw_model = (
    TRAINED_MODEL._orig_mod
    if hasattr(
        TRAINED_MODEL,
        "_orig_mod",
    )
    else TRAINED_MODEL
)

common_metadata = {
    "config": asdict(cfg),
    "features": list(FEATURES),
    "targets": list(TARGETS),
    "normalization_stats":
        TRAIN_DATASET.stats,
    "training_scope":
        "joint_full_model_initial_training",
    "later_daily_finetuning_scope":
        "participant_population_gate_transition",
    "runtime_onboard_scope":
        "separate_runtime_file",
    "encoder_type":
        cfg.encoder_type,
    "encoder_causal":
        cfg.encoder_causal,
    "decision_interval_seconds":
        cfg.decision_interval_seconds,
    "agent_count":
        cfg.num_agents,
    "agent_communication":
        "none_within_step",
    "transition_input":
        "weighted_aggregate_action_only",
}

torch.save(
    {
        **common_metadata,
        "model_state_dict":
            raw_model.state_dict(),
        "training_history":
            CHECKPOINT.get(
                "history",
                None,
            )
            if isinstance(
                CHECKPOINT,
                dict,
            )
            else None,
    },
    FULL_MODEL_CHECKPOINT_PATH,
)

torch.save(
    {
        **common_metadata,
        "encoder_state_dict":
            raw_model.encoder.state_dict(),
    },
    ENCODER_CHECKPOINT_PATH,
)

torch.save(
    {
        **common_metadata,
        "population_state_dict":
            raw_model.population.state_dict(),
        "gate_state_dict":
            raw_model.population.group_gate.state_dict(),
    },
    PARTICIPANT_CHECKPOINT_PATH,
)

torch.save(
    {
        **common_metadata,
        "transition_state_dict":
            raw_model.transition.state_dict(),
    },
    TRANSITION_CHECKPOINT_PATH,
)

print(
    "Saved full simulator:",
    FULL_MODEL_CHECKPOINT_PATH,
)

print(
    "Saved encoder:",
    ENCODER_CHECKPOINT_PATH,
)

print(
    "Saved participant population:",
    PARTICIPANT_CHECKPOINT_PATH,
)

print(
    "Saved transition model:",
    TRANSITION_CHECKPOINT_PATH,
)